# EEG Hackathon Student Network for Neuroscience - PsyFaKo Konferenz - Short Version


### 0. Imports & Setup
Import all necessary libraries and set the paths for the data directory.

In [ ]:
import mne
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import os
from mne.preprocessing import ICA, create_eog_epochs

mne.set_log_level('WARNING')  # reduce verbose output

print('MNE version:', mne.__version__)

### 1. Load the raw data for participant 1

Let's try to load the raw data for participant 1 now. Use the mne.io.read_raw_brainvision() function to read the .vhdr file. Afterwards show some basic information about the raw data, such as the duration of the recording and the number of channels. 
Optional: plot a section of the raw data (e.g. 10 seconds) to visually inspect it.

In [ ]:
data_dir = '../../data/hackathon_data/sub-1/eeg/sub-1_task-gonogo_eeg.vhdr' #adjust to your path
raw = mne.io.read_raw_brainvision(data_dir, preload=True)
ica_raw = raw.copy()

# show some basic information about the raw data
print('\n── Raw info ──────────────────')
print(raw.info)
print(f'Duration: {raw.times[-1]:.1f} s')

### 2. Optional: Bad Channel Rejection

Only do if you have progressed quickly so far

There are 2 main ways to mark bad channels in MNE - either you add them to the respective field in you data object, or - if you have a visualization of your data (with which you can interact - this can be a bit annoying to get to work (especially in a Jupyter notebook)) you can manually select channels to reject. 

In [ ]:
# plot the raw data to visually inspect it to identify potential bad channels

%matplotlib widget
mne.viz.set_browser_backend('matplotlib')

#alternative to widget is qt, if you want it in a popup window
#requires to have ipympl and ipywidgets installed in your environment!
#if you do not want interactive plots, replace widget with inline
#the interactive plot requires to have ipympl and ipywidgets installed in your environment!
#if you have an interactive plot, you can mark channels as bad by clicking on them (note: with this code this would not work, as we are only plotting a cropped copy of the actual data we are working with)
raw_short = raw.copy().crop(tmax=60)
raw_short.plot(n_channels=20, duration=10, title='Raw EEG Data - Participant 1',)

del raw_short

In [ ]:
#Other option: directly add the bad channels to your data. There are multiple ways to do this - here is one option:
montage = mne.channels.make_standard_montage('standard_1020') #we need to define the spatial arrangement of the electrodes, in case we want to do interpolation later
raw.set_montage(montage, on_missing='ignore')
ica_raw.set_montage(montage, on_missing='ignore')

#we need to mark the channels as bad in both the data that we want to use for filtering, and the one we fit the ICA on, as otherwise there will be a mismatch in the channels on which the ICA is fitted and applied.
raw.info['bads'].extend(['P10']) #This is just an example channel we identified previously as bad
ica_raw.info['bads'].extend(['P10'])


#Verify Channels have been marked as bad:
print(raw.info)


### 3. Filtering the data
In order to prepare the data for further analysis, we will now apply a band-pass filter to the raw data. This will help us to remove slow drifts and high-frequency noise from the data, which can interfere with our analysis. We will use a band-pass filter with a low cutoff frequency of 1 Hz and a high cutoff frequency of 30 Hz, which is a common choice for EEG data.
To see how the filter affects the data you should plot the power spectral density (PSD) of the raw data before filtering, then apply the band-pass filter and then compute and plot the PSD again to see how the filter has affected the data. This will allow you to visually inspect the effect of the filter on the frequency content of the data.


In [ ]:
plt.close('all')
# Before we filter we first look at what we are actually filtering out by plotting the power spectral density (PSD) of the raw data
spectrum = raw.compute_psd()
spectrum.plot(average=True, picks='data', exclude='bads', amplitude=False, show = False);

# Now we apply the band-pass filter to the raw data
raw.filter(l_freq=0.2, h_freq= 40)

In [ ]:
# After filtering, we can again look at the power spectral density to see how the filter has affected the data
spectrum_filtered = raw.compute_psd()
spectrum_filtered.plot(average=True, picks='data', exclude='bads', amplitude=False, show = False);

#we can also check how it affected the data info
print(raw.info)


### 4. Resampling the Data
After filtering, we will resample the data to a lower sampling rate. This can help to reduce the computational load for further analysis steps, while still retaining the relevant information in the data. We will resample the data to 256 Hz.
Importantly we need to include our events here, since the resampling can create small shifts in the data, which can lead to misalignment between the events and the data. By including the events in the resampling process, we can ensure that they remain properly aligned with the data after resampling.
To check what changed after resampling, you should again plot the power spectral density and compare it to the one before resampling, but after filtering.


In [ ]:
plt.close('all')

# Extract events (returns a tuple: events array + event_id mapping)
events, event_id = mne.events_from_annotations(raw)

# Resample raw dataset (annotations in raw are time-based and survive resampling automatically;
# therefore pass the sample-based events array so their indices get adjusted to the new sampling rate)
events = raw.resample(sfreq=256, events=events)

# Test whether the resampling worked by looking at the new sampling frequency
print(f'New sampling frequency: {raw.info["sfreq"]} Hz')

# May also look at the power spectral density again to see how the resampling has affected the data
# the maximum frequency that can be represented has changed! (Look up the Nyquist-Shannon Theorem :])
spectrum_resampled = raw.compute_psd()
spectrum_resampled.plot(average=True, picks='data', exclude='bads', amplitude=False, show = False);

### 5. Re-Referencing of the Data
An EEG electrode does **not** measure absolute voltage — it always measures the **difference**
between itself and a chosen **reference electrode**. The online-reference is set during recording
(e.g. Cz, one mastoid). Depending on the activity at that electrode, it may create a systematic bias.
 
**Average reference** 
The standard solution in ERP research is the **average reference**: re-reference every channel
to the mean of *all* channels simultaneously:

In [ ]:
raw.set_eeg_reference(ref_channels='average', projection = False) 
ica_raw.set_eeg_reference(ref_channels='average', projection = False)

print(raw.info)

### 6. ICA - Independant Component Analysis
**Independent Component Analysis** decomposes the mixed EEG signal into a set of statistically
**independent source signals** (components). Artifacts like eye blinks have very characteristic
patterns and ICA isolates these into their own components.
We remove those components and reconstruct the signal from the rest.

Reminder: You fit the ICA on a 1Hz High Pass filtered copy of your original data - not the one you have already filtered! - and then apply the result back to your filtered data, after rejecting potential bad components.


Bonus Info: How to identify artifact components (ICLabel criteria)
Realistically you will not have time for this - in doubt just remove a random component, so you know how to do it
 
Use **three complementary plots** — topographic maps, time series, and correlation scores:
 
| Artifact | Topography | Time series | Power spectrum |
|---|---|---|---|
| Eye blink | Large symmetric loading at Fp1/Fp2 | Regular large spikes | Most power <5 Hz |
| Eye movement | Asymmetric frontal (left/right) | Step-function pattern | Most power <5 Hz |
| Heartbeat | Near-linear gradient | Regular QRS complexes (~1/s) | No clear peaks |
| Muscle | Focal, shallow (scalp edge) | Non-stationary bursts | Broadband >20 Hz |
| Brain  | Dipolar map, RV <15% | ERP if epoched | 1/f + peak 5–30 Hz |
 
*(Based on ICLabel taxonomy — labeling.ucsd.edu/tutorial/labels)*
 

In [ ]:
plt.close('all')
ica_raw.filter(l_freq=1.0, h_freq=None)
ica = ICA(n_components=12, max_iter='auto', random_state=17)
ica.fit(ica_raw)


%matplotlib widget
mne.viz.set_browser_backend('matplotlib')
# Topographic maps of all components
ica.plot_components()

# Time-series of each component — plotted on the real (non-1-Hz) data
ica.plot_sources(raw);

In [ ]:
%matplotlib widget
mne.viz.set_browser_backend('matplotlib')

bad_components = [9,10,11] #put the index of components you want to exclude here

 
raw_clean = ica.apply(raw.copy(), exclude=bad_components)


# Get the original and cleaned data for a specific channel
original_data = raw.get_data(picks='Fz')[0]  # Pick a channel
cleaned_data = raw_clean.get_data(picks='Fz')[0]

# Calculate the difference
difference = original_data - cleaned_data

print(f"Mean difference: {np.mean(np.abs(difference)):.6f}")
print(f"Max difference: {np.max(np.abs(difference)):.6f}")
print(f"Variance removed: {(1 - np.var(cleaned_data)/np.var(original_data))*100:.2f}%")



### 7. Optional - Interpolation
This was not explicitly covered in our input - interpolation is a method with which you can replace the signal on channels that you have previously rejected (Step 2) with the estimated activity based on weigthed estimation of the activity recorded at neighboring electrodes.

In [ ]:
raw_clean.interpolate_bads(reset_bads=True, method = 'spline')
print(raw_clean.info)

### 8. Epoch your Data
Event markers were recorded as annotations. Convert them to an array MNE can use for epoching.

In [ ]:
# Extract events 
events, event_id = mne.events_from_annotations(raw_clean)

print(event_id)
print(f'\nTotal events: {len(events)}')
print('\nFirst 10 rows (sample, 0, event_code):')
print(events[:10])

# Triggers 1, 2, 3 = Go stimuli; Trigger 4 = NoGo stimulus.
# The definition of which Trigger corresponds to what kind of event is something you need to decide when creating your experimental paradigm
# Use hierarchical naming (Go/1, Go/2, Go/3) so that epochs['Go']
# automatically selects all three sub-conditions.
event_id_select = {'Go/1': 1, 'Go/2': 2, 'Go/3': 3, 'NoGo': 4}

epochs = mne.Epochs(
    raw_clean, events, event_id=event_id_select,
    tmin=-0.1, tmax=0.6,
    baseline=None, preload=True, verbose=False
)

print(epochs)
print(f'\nGo:   {len(epochs["Go"])} epochs')
print(f'NoGo: {len(epochs["NoGo"])} epochs')

### 9.Baseline Correction
 
EEG signals drift — the overall voltage level shifts gradually over time.
This means the starting voltage of each epoch is slightly different,
making cross-trial comparisons unfair.
 
**Baseline correction** removes this drift

In [ ]:
# Baseline Correction (−100 to 0 ms)
epochs.apply_baseline(baseline=(-0.1, 0))

# Sanity check: mean amplitude in the baseline window should be ≈ 0
baseline_check = epochs.copy().crop(tmin=-0.1, tmax=0)
mean_uv = baseline_check.get_data().mean() * 1e6
print(f'Mean amplitude in baseline window: {mean_uv:.6f} µV  (should be ≈ 0)')

### 10. Save your processed file
Save the processed data as .fif file.

Epoch files must end with `-epo.fif` or `_epo.fif` (MNE naming convention).

In [ ]:
epochs.save('../../data/hackathon_data/sub-1_task-gonogo_epo.fif', overwrite=True)